# 01. ACOS Setup, Pretrained Model Caching & Exploratory Data Analysis (EDA)

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook handles the initial setup for running ACOS on **Google Colab** (or Local environment), checks GPU acceleration, downloads and caches the pretrained `bert-base-uncased` model assets locally (to eliminate legacy S3 download failures), and performs comprehensive Exploratory Data Analysis (EDA) with publication-quality visualizations and CSV statistical exports.

## 1. Environment Setup & Dependency Installation
Install required dependencies (`torchcrf`, `transformers`, `huggingface_hub`, `seaborn`, `scikit-learn`).

In [ ]:
# Check if running on Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("🚀 Running in Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("💻 Running in Local environment.")

# Install dependencies
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3

import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU Availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected. Running on CPU.")

## 2. Directory Navigation & Path Initialization

In [ ]:
# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 3. Import colab_utils with fallback download
try:
    from colab_utils import setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda

print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")


## 3. Initialize Timestamped Session Directory (`DDMMYYYY_HMS`)
Every session automatically creates an isolated directory inside `results/` for storing plots, CSV tables, model checkpoints, and execution logs.

In [ ]:
# Choose domain: 'rest16' (Restaurant-ACOS) or 'laptop' (Laptop-ACOS)
DOMAIN = "rest16"

results_base = os.path.join(base_project_dir, "results")
session_dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)

print("Directory structure:")
for k, v in session_dirs.items():
    print(f"  - {k}: {v}")

## 4. Download & Cache Pretrained BERT Model (`bert-base-uncased`)
Downloads `config.json`, `pytorch_model.bin`, and `vocab.txt` directly from HuggingFace Hub to local cache `./bert_base_uncased`.

In [ ]:
bert_cache_dir = os.path.join(base_project_dir, "bert_base_uncased")
download_bert_pretrained(target_dir=bert_cache_dir)

# Verify files
for f in ["config.json", "pytorch_model.bin", "vocab.txt"]:
    fpath = os.path.join(bert_cache_dir, f)
    assert os.path.exists(fpath), f"Missing BERT file: {fpath}"
    print(f"✅ {f} ({os.path.getsize(fpath)/1024/1024:.2f} MB)")

## 5. Exploratory Data Analysis (EDA) & Visualization
Perform deep statistical analysis on `Restaurant-ACOS` (`rest16`) and `Laptop-ACOS` (`laptop`) datasets:
- Sentence and Quadruple counts per split (Train / Dev / Test)
- Explicit vs. Implicit Aspect and Opinion proportions
- Top Aspect Categories & Sentiment Polarity distributions
- Automatic export to high-resolution PNG plots (`plots/`) and CSV files (`csv/`).

In [ ]:
data_root = os.path.join(base_project_dir, "data")

# Analyze chosen domain
df_stats, df_records = analyze_and_plot_eda(
    data_dir=data_root,
    domain=DOMAIN,
    output_plots_dir=session_dirs["plots"],
    output_csv_dir=session_dirs["csv"]
)

print(f"\n=== [{DOMAIN.upper()}] Dataset Summary Table ===")
display(df_stats)

### Sample Data Preview with Implicit/Explicit Flags

In [ ]:
if df_records is not None and not df_records.empty:
    print(f"Total Quadruple Records Analyzed: {len(df_records)}")
    display(df_records.head(10))
    
    # Summary of Implicit Aspect & Opinion presence
    imp_asp_pct = (df_records["Is_Implicit_Aspect"].sum() / len(df_records)) * 100
    imp_opi_pct = (df_records["Is_Implicit_Opinion"].sum() / len(df_records)) * 100
    print(f"\n🔍 Implicit Aspects: {df_records['Is_Implicit_Aspect'].sum()} ({imp_asp_pct:.2f}%)")
    print(f"🔍 Implicit Opinions: {df_records['Is_Implicit_Opinion'].sum()} ({imp_opi_pct:.2f}%)")

### Display Generated Visualization Charts
Visualizations are rendered inline and saved to `plots/`.

In [ ]:
from IPython.display import Image, display

plot1 = os.path.join(session_dirs["plots"], "01_eda_dataset_distribution.png")
plot2 = os.path.join(session_dirs["plots"], "02_eda_category_sentiment.png")

if os.path.exists(plot1):
    print("📊 Plot 1: Dataset Composition & Explicit vs. Implicit Distribution")
    display(Image(plot1))
    
if os.path.exists(plot2):
    print("📊 Plot 2: Aspect Category & Sentiment Polarity Breakdown")
    display(Image(plot2))

## 6. Summary of Exported Artifacts
All outputs from this step are safely persisted in the timestamped session directory:

In [ ]:
print("📁 Generated CSV Files:")
for f in os.listdir(session_dirs["csv"]):
    print(f"  - {os.path.join(session_dirs['csv'], f)}")

print("\n📁 Generated Plots:")
for f in os.listdir(session_dirs["plots"]):
    print(f"  - {os.path.join(session_dirs['plots'], f)}")

print("\n✨ Setup & EDA completed successfully. Proceed to '02_ACOS_Step1_Aspect_Opinion_Extraction.ipynb'!")